In [40]:
import pandas as pd

In [41]:
# Leer el archivo copiado
df = pd.read_parquet("sp500_history_copy.parquet")

In [42]:
#Explora las columnas disponibles y sus tipos.
df.columns.tolist()
df.dtypes

date                datetime64[ns]
symbol                         str
assetid                      int64
security_name                  str
sector                         str
industry                       str
subsector                      str
in_sp500                     int32
open                       float32
high                       float32
low                        float32
close                      float32
volume                     float32
unadjusted_close           float32
dtype: object

In [43]:
#Muestra una vista rapida (head) 
df.head()

,date,symbol,assetid,security_name,sector,industry,subsector,in_sp500,open,high,low,close,volume,unadjusted_close
0,1999-11-18,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,27.188307,29.877260,23.901808,25.545057,74862288.0,42.7500
1,1999-11-19,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,25.657097,25.694445,23.789768,24.349968,18236110.0,40.7500
2,1999-11-22,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,24.686087,26.067909,23.939156,26.067909,7874048.5,43.6250
3,1999-11-23,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,25.395672,26.067909,24.051195,24.051195,7153099.0,40.2500
4,1999-11-24,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,0,23.976501,25.059551,23.901808,24.536699,5797720.5,41.0625


In [44]:
#Validaciones iniciales

# Rangos de fechas
print("Rango de fechas:")   
print(f"Fecha mínima: {df['date'].min()}, Fecha máxima: {df['date'].max()}")

Rango de fechas:
Fecha mínima: 1990-01-02 00:00:00, Fecha máxima: 2026-01-30 00:00:00


In [45]:
# Filtrar para poder iniciar backtest en 2015-10-31 con lag=1 y R_12:
# se requiere al menos historial desde 2014-09-30.
df = df[df['date'] >= '2013-01-01'].copy()
print(f"Rango de fechas despues del filtro: {df['date'].min()} a {df['date'].max()}")

Rango de fechas despues del filtro: 2013-01-02 00:00:00 a 2026-01-30 00:00:00


In [46]:
# Duplicados exactos por fecha y ticker (misma fecha + mismo símbolo)
counts = df.groupby(['date', 'symbol']).size()
dup_exact = counts[counts > 1]

if dup_exact.empty:
    print("No hay duplicados exactos (fecha, ticker).")
else:
    print(f"Hay {len(dup_exact)} duplicados exactos (fecha, ticker).")
    print(dup_exact.head(20))

# Múltiples símbolos para la misma empresa en la misma fecha
# Usamos 'security_name' como proxy de empresa
company_col = 'security_name'

sym_nunique = df.groupby(['date', company_col])['symbol'].nunique()
multisym = sym_nunique[sym_nunique > 1]

if multisym.empty:
    print("No hay empresas con múltiples símbolos en la misma fecha.")
else:
    print(f"Hay {len(multisym)} combinaciones (fecha, empresa) con múltiples símbolos.")
    symbols_per = (df.groupby(['date', company_col])['symbol']
                     .apply(lambda x: ', '.join(sorted(set(x)))))
    detalle = symbols_per[multisym.index].reset_index(name='symbols')
    detalle['n_symbols'] = multisym.values
    print(detalle.head(20).to_string(index=False))

No hay duplicados exactos (fecha, ticker).
No hay empresas con múltiples símbolos en la misma fecha.


In [47]:
# Resolver duplicados por date + assetid:
# conservar la fila con mayor volume.
entity_keys = ['date', 'assetid']

dup_mask = df.duplicated(subset=entity_keys, keep=False)
rows_in_dup_groups = int(dup_mask.sum())
groups_with_dups = int(df.loc[dup_mask].groupby(entity_keys).ngroups)

print(f"Filas en grupos duplicados (date, assetid): {rows_in_dup_groups}")
print(f"Grupos duplicados (date, assetid): {groups_with_dups}")

if rows_in_dup_groups > 0:
    # Orden estable para desempates: mayor volumen, luego symbol asc
    df = (
        df.sort_values(['date', 'assetid', 'volume', 'symbol'], ascending=[True, True, False, True], na_position='last')
          .drop_duplicates(subset=entity_keys, keep='first')
          .reset_index(drop=True)
    )

remaining_dups = int(df.duplicated(subset=entity_keys).sum())
print(f"Duplicados restantes (date, assetid): {remaining_dups}")

# Validacion adicional: mismo date + symbol repetido
dup_symbol = int(df.duplicated(subset=['date', 'symbol']).sum())
print(f"Duplicados exactos restantes (date, symbol): {dup_symbol}")


Filas en grupos duplicados (date, assetid): 0
Grupos duplicados (date, assetid): 0
Duplicados restantes (date, assetid): 0
Duplicados exactos restantes (date, symbol): 0


In [48]:
df.to_pickle("sp500_history_filtered.pkl")